# Đề tài 10 · Nhận dạng mẫu: Phân loại Âm thanh & Phát hiện Giọng nói Nhân bản Zero-Shot
## Thực nghiệm 1.2: Quy trình Sinh Ngữ liệu Đối chứng Đa Nguồn (6.300 Audio Clips Synthesis)

**Nhóm nghiên cứu: Nhóm 05 · Chuyên ngành Khoa học Máy tính / Trí tuệ Nhân tạo**

### 1. Giới thiệu & Khung Đề tài
Trong quy trình đánh giá tính bền vững và khả năng nhận diện âm thanh nhân bản zero-shot, việc thu thập ngữ liệu đối chứng đa dạng và có kiểm soát chặt chẽ là điều kiện tiên quyết. Notebook này thiết lập pipeline tự động sinh 6.300 bản ghi âm tổng hợp đối sánh trên 300 câu ngữ âm tiếng Việt đại diện.

| Phân loại bài toán | Mục tiêu nhận dạng | Số lớp phân loại ($K$) | Tập dữ liệu mục tiêu |
|:---|:---|:---:|:---|
| **Bài toán T-B** | Nhận diện hệ thống sinh âm thanh (Generator Attribution) | **3 lớp** | OmniVoice gốc, OmDiOm-ft, VieNeu-TTS |
| **Bài toán T-C** | Nhận diện giọng nói tổng hợp theo điều kiện tham chiếu (Speaker Verification) | **7 lớp** | 7 phong cách giọng tham chiếu zero-shot |
| **Bài toán T-A** | Phân loại âm thanh Thật vs Giả mạo (Deepfake Detection) | **2 lớp** | Thực nghiệm độc lập trên ngữ liệu chuẩn VIVOS |

### 2. Thiết kế Ma trận Dữ liệu
- Ma trận thực nghiệm bao gồm: **300 câu chuẩn × 7 mẫu giọng tham chiếu × 3 hệ thống sinh = 6.300 clips**.
- Quy chuẩn xử lý tín hiệu:
  - Mã hóa tín hiệu: PCM 16-bit, tần số lấy mẫu đồng bộ 16.000 Hz, kênh đơn (Mono).
  - Chuẩn hóa biên độ tín hiệu âm lượng theo chuẩn Peak Normalization (-1.0 dBFS).
  - Khởi tạo mẫu tham chiếu định dạng 16 kHz WAV trích xuất từ các véc-tơ đặc trưng âm học chuẩn.

### 3. Quy trình Thực thi Khoa học (Pipeline Architecture)
1. Nạp và giải mã véc-tơ đặc trưng âm học 7 giọng tham chiếu tiếng Việt.
2. Khởi tạo mô hình nền tảng tổng hợp tiếng nói đa ngôn ngữ OmniVoice và biến thể tinh chỉnh OmDiOm-ft.
3. Khởi tạo mô hình đối sánh VieNeu-TTS dựa trên tệp tham chiếu WAV tương ứng.
4. Thực thi vòng lặp suy luận (Inference Loop) theo cơ chế lưu vết (Checkpoint & Resume) đảm bảo độ tin cậy.
5. Đánh giá tính hợp thức của tín hiệu xuất bản qua bảng siêu dữ liệu (`manifest.csv`).
6. Xây dựng tài liệu báo cáo phân tích thăm dò dữ liệu (EDA) và trình phát đối sánh âm học nhúng.

### 4. Sản phẩm Đầu ra (Target Artifacts)
- Thư mục âm thanh: `pa2/audio/<bo_sinh>/<giong>/<cau_id>.flac` (6.300 tệp âm thanh FLAC).
- Thư mục giọng chuẩn: `pa2/ref/<giong>.wav` (7 tệp WAV tham chiếu chuẩn hóa).
- Bảng nhãn: `pa2/manifest.csv` (8 thuộc tính siêu dữ liệu chuẩn hóa).
- Báo cáo phân tích: `BAO_CAO.html` (Báo cáo tự chứa nhúng 17 đồ thị phân tích âm học và trình phát so sánh).


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 1 · CẤU HÌNH — chỉ sửa ở đây
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, glob, json, time, shutil, subprocess, importlib, importlib.util
import numpy as np

GOC_GIONG  = "/kaggle/input/datasets/tts-pro-vip"   # 7 tệp .pt
DUONG_FT   = ""            # trống = tự dò trọng số OmDiOm-ft; hoặc ghi đường dẫn
RA         = "/kaggle/working/pa2"
DIR_AM     = os.path.join(RA, "audio")
DIR_REF    = os.path.join(RA, "ref")

CHAY_THU   = False         # ← True: chỉ 3 câu (63 clip) để kiểm tra
SO_CAU     = 300
GIO_TOI_DA = 10.5          # tự dừng trước khi Kaggle cắt phiên 12 h

NUM_STEP   = 32            # giữ 32 như PHIÊN 3 để so sánh được
DINH_DANG  = "flac"
SEED       = 1234

GIONG = ["ke", "u_linh", "giong_nu_tre", "giong_ong_gia",
         "giong_trai_tre", "giong_tre_em", "tran_thy"]

CAU_REF = ("Hôm nay trời đẹp, tôi muốn đi dạo một vòng quanh hồ và kể cho bạn nghe "
           "vài câu chuyện nhỏ. Giọng đọc này cần đủ dài để người nghe cảm nhận được "
           "âm sắc, nhịp điệu và cách lên xuống ở cuối mỗi câu.")

for d in (RA, DIR_AM, DIR_REF):
    os.makedirs(d, exist_ok=True)

import torch
print("Python", sys.version.split()[0], "· torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      ("· " + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 2 · BỘ CÂU CHUẨN — 300 câu, 50 mỗi thanh, chia sẵn 240 train / 60 test
# ═══════════════════════════════════════════════════════════════════════════
# Nhúng thẳng vào notebook: KHÔNG phải upload thêm dataset nào.
# Mỗi dòng: (cau_id, noi_dung, thanh, tap, so_am_tiet)
CAU_CHUAN = [
    ('S0001', 'Ba ra sân sau ngắm hoa.', 'ngang', 'test', 6),
    ('S0002', 'Nam ra sân sau ngắm hoa.', 'ngang', 'train', 6),
    ('S0003', 'Lan ra sân sau ngắm hoa.', 'ngang', 'test', 6),
    ('S0004', 'An ra sân sau ngắm hoa.', 'ngang', 'train', 6),
    ('S0005', 'Nga ra sân sau ngắm hoa.', 'ngang', 'train', 6),
    ('S0006', 'Ba mang hoa ra sân.', 'ngang', 'test', 5),
    ('S0007', 'Ba mang cam ra sân.', 'ngang', 'train', 5),
    ('S0008', 'Ba mang rau ra sân.', 'ngang', 'test', 5),
    ('S0009', 'Ba mang dao ra sân.', 'ngang', 'train', 5),
    ('S0010', 'Ba mang nem ra sân.', 'ngang', 'train', 5),
    ('S0011', 'Nam đi qua ga xa.', 'ngang', 'train', 5),
    ('S0012', 'Lan đi qua ga xa.', 'ngang', 'train', 5),
    ('S0013', 'An đi qua ga xa.', 'ngang', 'train', 5),
    ('S0014', 'Nga đi qua ga xa.', 'ngang', 'train', 5),
    ('S0015', 'Ba đi qua ga xa.', 'ngang', 'train', 5),
    ('S0016', 'Ta ca vang qua sân.', 'ngang', 'train', 5),
    ('S0017', 'Ta ca vang qua ga.', 'ngang', 'test', 5),
    ('S0018', 'Ta ca vang qua ao.', 'ngang', 'train', 5),
    ('S0019', 'Ta ca vang qua ban công.', 'ngang', 'train', 6),
    ('S0020', 'Ta ca vang qua sân sau.', 'ngang', 'train', 6),
    ('S0021', 'Ba mua cam cho Lan.', 'ngang', 'train', 5),
    ('S0022', 'Ba mua hoa cho Lan.', 'ngang', 'train', 5),
    ('S0023', 'Ba mua rau cho Lan.', 'ngang', 'train', 5),
    ('S0024', 'Ba mua dao cho Lan.', 'ngang', 'train', 5),
    ('S0025', 'Ba mua nem cho Lan.', 'ngang', 'train', 5),
    ('S0026', 'Nam đi ngang qua ga.', 'ngang', 'train', 5),
    ('S0027', 'Nam đi ngang qua sân.', 'ngang', 'test', 5),
    ('S0028', 'Nam đi ngang qua ao.', 'ngang', 'train', 5),
    ('S0029', 'Nam đi ngang qua bờ ao.', 'ngang', 'train', 6),
    ('S0030', 'Nam đi ngang qua sân sau.', 'ngang', 'train', 6),
    ('S0031', 'Hoa lan nở quanh sân.', 'ngang', 'train', 5),
    ('S0032', 'Hoa lan nở quanh ao.', 'ngang', 'train', 5),
    ('S0033', 'Hoa lan nở quanh ban công.', 'ngang', 'test', 6),
    ('S0034', 'Hoa lan nở quanh sân sau.', 'ngang', 'train', 6),
    ('S0035', 'Hoa lan nở quanh ga xa.', 'ngang', 'train', 6),
    ('S0036', 'Con chim bay qua sân.', 'ngang', 'train', 5),
    ('S0037', 'Con chim bay qua ao.', 'ngang', 'train', 5),
    ('S0038', 'Con chim bay qua ga.', 'ngang', 'train', 5),
    ('S0039', 'Con chim bay qua ban công.', 'ngang', 'train', 6),
    ('S0040', 'Con chim bay qua sân sau.', 'ngang', 'train', 6),
    ('S0041', 'Ta đưa hoa ra ban công.', 'ngang', 'train', 6),
    ('S0042', 'Ta đưa hoa ra sân sau.', 'ngang', 'test', 6),
    ('S0043', 'Ta đưa hoa ra ga xa.', 'ngang', 'train', 6),
    ('S0044', 'Ta đưa hoa ra sân nhà.', 'ngang', 'train', 6),
    ('S0045', 'Ta đưa hoa ra bờ ao.', 'ngang', 'train', 6),
    ('S0046', 'Ba kê ghế ra sân.', 'ngang', 'train', 5),
    ('S0047', 'Ba kê ghế ra ban công.', 'ngang', 'test', 6),
    ('S0048', 'Ba kê ghế ra sân sau.', 'ngang', 'train', 6),
    ('S0049', 'Ba kê ghế ra bờ ao.', 'ngang', 'test', 6),
    ('S0050', 'Ba kê ghế ra ga xa.', 'ngang', 'train', 6),
    ('S0051', 'Bà Hà về nhà rồi.', 'huyen', 'train', 5),
    ('S0052', 'Bà Hà về làng rồi.', 'huyen', 'train', 5),
    ('S0053', 'Bà Hà về phòng rồi.', 'huyen', 'train', 5),
    ('S0054', 'Bà Hà về quê nhà rồi.', 'huyen', 'train', 6),
    ('S0055', 'Bà Hà về miền quê rồi.', 'huyen', 'train', 6),
    ('S0056', 'Ánh đèn vàng soi thềm nhà.', 'huyen', 'train', 6),
    ('S0057', 'Ánh đèn mờ soi thềm nhà.', 'huyen', 'train', 6),
    ('S0058', 'Ánh đèn hồng soi thềm nhà.', 'huyen', 'train', 6),
    ('S0059', 'Ánh đèn dầu soi thềm nhà.', 'huyen', 'train', 6),
    ('S0060', 'Ánh đèn lồng soi thềm nhà.', 'huyen', 'test', 6),
    ('S0061', 'Người già thường về làng.', 'huyen', 'train', 5),
    ('S0062', 'Người già thường về nhà.', 'huyen', 'test', 5),
    ('S0063', 'Người già thường về miền quê.', 'huyen', 'train', 6),
    ('S0064', 'Người già thường về quê nhà.', 'huyen', 'train', 6),
    ('S0065', 'Người già thường về làng xưa.', 'huyen', 'train', 6),
    ('S0066', 'Bà mang trà về nhà.', 'huyen', 'train', 5),
    ('S0067', 'Bà mang chè về nhà.', 'huyen', 'train', 5),
    ('S0068', 'Bà mang quà về nhà.', 'huyen', 'train', 5),
    ('S0069', 'Bà mang dầu về nhà.', 'huyen', 'train', 5),
    ('S0070', 'Bà mang tiền về nhà.', 'huyen', 'train', 5),
    ('S0071', 'Mùa hè, mẹ về nhà.', 'huyen', 'test', 5),
    ('S0072', 'Mùa hè, mẹ về làng.', 'huyen', 'test', 5),
    ('S0073', 'Mùa hè, mẹ về quê nhà.', 'huyen', 'test', 6),
    ('S0074', 'Mùa hè, mẹ về miền Nam.', 'huyen', 'train', 6),
    ('S0075', 'Mùa hè, mẹ về miền Trung.', 'huyen', 'train', 6),
    ('S0076', 'Con đường làng đầy lá vàng.', 'huyen', 'train', 6),
    ('S0077', 'Con đường làng đầy sương mờ.', 'huyen', 'train', 6),
    ('S0078', 'Con đường làng đầy rêu mềm.', 'huyen', 'train', 6),
    ('S0079', 'Con đường làng đầy bụi đường.', 'huyen', 'train', 6),
    ('S0080', 'Con đường làng đầy ánh đèn.', 'huyen', 'train', 6),
    ('S0081', 'Chiều về, bà ngồi bên thềm.', 'huyen', 'train', 6),
    ('S0082', 'Chiều về, bà ngồi bên hè.', 'huyen', 'train', 6),
    ('S0083', 'Chiều về, bà ngồi bên hiên nhà.', 'huyen', 'train', 7),
    ('S0084', 'Chiều về, bà ngồi bên đường làng.', 'huyen', 'train', 7),
    ('S0085', 'Chiều về, bà ngồi bên bờ tường.', 'huyen', 'train', 7),
    ('S0086', 'Ngoài hè, người già nằm chờ.', 'huyen', 'test', 6),
    ('S0087', 'Ngoài hè, người già nằm mơ.', 'huyen', 'train', 6),
    ('S0088', 'Ngoài hè, người già nằm nghe chèo.', 'huyen', 'train', 7),
    ('S0089', 'Ngoài hè, người già nằm bên thềm.', 'huyen', 'train', 7),
    ('S0090', 'Ngoài hè, người già nằm nhìn về làng.', 'huyen', 'test', 8),
    ('S0091', 'Mẹ đem chè về nhà.', 'huyen', 'test', 5),
    ('S0092', 'Mẹ đem trà về nhà.', 'huyen', 'train', 5),
    ('S0093', 'Mẹ đem dầu về nhà.', 'huyen', 'test', 5),
    ('S0094', 'Mẹ đem tiền về nhà.', 'huyen', 'train', 5),
    ('S0095', 'Mẹ đem đèn về nhà.', 'huyen', 'train', 5),
    ('S0096', 'Bà kể về mùa hè, về làng quê.', 'huyen', 'train', 8),
    ('S0097', 'Bà kể về mùa hè, về đường làng.', 'huyen', 'train', 8),
    ('S0098', 'Bà kể về mùa hè, về miền xa.', 'huyen', 'train', 8),
    ('S0099', 'Bà kể về mùa hè, về quê nhà.', 'huyen', 'train', 8),
    ('S0100', 'Bà kể về mùa hè, về bờ đê.', 'huyen', 'test', 8),
    ('S0101', 'Bé thích uống nước mát.', 'sac', 'test', 5),
    ('S0102', 'Phúc thích uống nước mát.', 'sac', 'train', 5),
    ('S0103', 'Bích thích uống nước mát.', 'sac', 'train', 5),
    ('S0104', 'Tú thích uống nước mát.', 'sac', 'train', 5),
    ('S0105', 'Lý thích uống nước mát.', 'sac', 'train', 5),
    ('S0106', 'Bé thích sách mới, bút sắc.', 'sac', 'train', 6),
    ('S0107', 'Bé thích sách mới, bút tốt.', 'sac', 'train', 6),
    ('S0108', 'Bé thích sách mới, bút trắng.', 'sac', 'train', 6),
    ('S0109', 'Bé thích sách mới, bút mới.', 'sac', 'train', 6),
    ('S0110', 'Bé thích sách mới, bút đẹp.', 'sac', 'test', 6),
    ('S0111', 'Sáng nay, chúng cháu đến lớp, thích hát.', 'sac', 'train', 8),
    ('S0112', 'Sáng nay, chúng cháu đến lớp, thích múa.', 'sac', 'train', 8),
    ('S0113', 'Sáng nay, chúng cháu đến lớp, thích vẽ.', 'sac', 'train', 8),
    ('S0114', 'Sáng nay, chúng cháu đến lớp, thích đọc sách.', 'sac', 'train', 9),
    ('S0115', 'Sáng nay, chúng cháu đến lớp, thích học toán.', 'sac', 'train', 9),
    ('S0116', 'Chúc mừng năm mới, chúc bạn thành công.', 'sac', 'train', 8),
    ('S0117', 'Chúc mừng năm mới, chúc bạn sức khỏe tốt.', 'sac', 'test', 9),
    ('S0118', 'Chúc mừng năm mới, chúc bạn phát tài.', 'sac', 'train', 8),
    ('S0119', 'Chúc mừng năm mới, chúc bạn may mắn.', 'sac', 'train', 8),
    ('S0120', 'Chúc mừng năm mới, chúc bạn tiến bộ.', 'sac', 'test', 8),
    ('S0121', 'Tiếng sáo réo rắt khắp xóm.', 'sac', 'train', 6),
    ('S0122', 'Tiếng sáo réo rắt khắp phố.', 'sac', 'train', 6),
    ('S0123', 'Tiếng sáo réo rắt khắp xứ.', 'sac', 'train', 6),
    ('S0124', 'Tiếng sáo réo rắt khắp xóm nhỏ.', 'sac', 'train', 7),
    ('S0125', 'Tiếng sáo réo rắt khắp phố nhỏ.', 'sac', 'train', 7),
    ('S0126', 'Chiếc bánh nóng, nước mát.', 'sac', 'train', 5),
    ('S0127', 'Chiếc bánh nóng, nước nóng.', 'sac', 'train', 5),
    ('S0128', 'Chiếc bánh nóng, nước sốt.', 'sac', 'test', 5),
    ('S0129', 'Chiếc bánh nóng, nước chấm.', 'sac', 'train', 5),
    ('S0130', 'Chiếc bánh nóng, nước sấu.', 'sac', 'train', 5),
    ('S0131', 'Phúc thích đọc sách tiếng Việt.', 'sac', 'train', 6),
    ('S0132', 'Phúc thích đọc sách tiếng Pháp.', 'sac', 'test', 6),
    ('S0133', 'Phúc thích đọc sách tiếng Thái.', 'sac', 'train', 6),
    ('S0134', 'Phúc thích đọc sách tiếng Nhật.', 'sac', 'test', 6),
    ('S0135', 'Phúc thích đọc sách tiếng Anh.', 'sac', 'test', 6),
    ('S0136', 'Sáng sớm, bé thích gió mát.', 'sac', 'train', 6),
    ('S0137', 'Sáng sớm, Phúc thích gió mát.', 'sac', 'test', 6),
    ('S0138', 'Sáng sớm, Bích thích gió mát.', 'sac', 'train', 6),
    ('S0139', 'Sáng sớm, Tú thích gió mát.', 'sac', 'train', 6),
    ('S0140', 'Sáng sớm, Lý thích gió mát.', 'sac', 'train', 6),
    ('S0141', 'Chúng cháu thích món súp nóng.', 'sac', 'train', 6),
    ('S0142', 'Chúng cháu thích món súp bí.', 'sac', 'train', 6),
    ('S0143', 'Chúng cháu thích món súp nấm.', 'sac', 'train', 6),
    ('S0144', 'Chúng cháu thích món súp cá.', 'sac', 'train', 6),
    ('S0145', 'Chúng cháu thích món súp thịt.', 'sac', 'train', 6),
    ('S0146', 'Ánh nắng chiếu sáng mái ngói.', 'sac', 'test', 6),
    ('S0147', 'Ánh nắng chiếu sáng mái tôn.', 'sac', 'train', 6),
    ('S0148', 'Ánh nắng chiếu sáng mái sắt.', 'sac', 'train', 6),
    ('S0149', 'Ánh nắng chiếu sáng mái kính.', 'sac', 'train', 6),
    ('S0150', 'Ánh nắng chiếu sáng mái hiên.', 'sac', 'train', 6),
    ('S0151', 'Bạn hỏi mẹ ở cửa sổ.', 'hoi', 'train', 6),
    ('S0152', 'Cô hỏi mẹ ở cửa sổ.', 'hoi', 'train', 6),
    ('S0153', 'Bà hỏi mẹ ở cửa sổ.', 'hoi', 'train', 6),
    ('S0154', 'Chị hỏi mẹ ở cửa sổ.', 'hoi', 'train', 6),
    ('S0155', 'Cậu hỏi mẹ ở cửa sổ.', 'hoi', 'train', 6),
    ('S0156', 'Cửa sổ nhỏ, cô hỏi nhỏ.', 'hoi', 'train', 6),
    ('S0157', 'Cửa sổ cũ, cô hỏi nhỏ.', 'hoi', 'train', 6),
    ('S0158', 'Cửa sổ mở, cô hỏi nhỏ.', 'hoi', 'train', 6),
    ('S0159', 'Cửa sổ hở, cô hỏi nhỏ.', 'hoi', 'train', 6),
    ('S0160', 'Cửa sổ rộng, cô hỏi nhỏ.', 'hoi', 'train', 6),
    ('S0161', 'Cô hỏi thử, chỗ nghỉ ở đâu?', 'hoi', 'train', 7),
    ('S0162', 'Cô hỏi thử, cửa sổ ở đâu?', 'hoi', 'train', 7),
    ('S0163', 'Cô hỏi thử, phòng nhỏ ở đâu?', 'hoi', 'train', 7),
    ('S0164', 'Cô hỏi thử, chỗ gửi ở đâu?', 'hoi', 'train', 7),
    ('S0165', 'Cô hỏi thử, cửa cũ ở đâu?', 'hoi', 'test', 7),
    ('S0166', 'Thử sửa cửa sổ nhỏ.', 'hoi', 'test', 5),
    ('S0167', 'Thử sửa cửa sổ cũ.', 'hoi', 'train', 5),
    ('S0168', 'Thử sửa cửa sổ mở.', 'hoi', 'test', 5),
    ('S0169', 'Thử sửa cửa sổ hở.', 'hoi', 'train', 5),
    ('S0170', 'Thử sửa cửa sổ rộng.', 'hoi', 'train', 5),
    ('S0171', 'Cô kể chuyện, hỏi thử về chỗ nghỉ.', 'hoi', 'train', 8),
    ('S0172', 'Cô kể chuyện, hỏi thử về cửa sổ.', 'hoi', 'train', 8),
    ('S0173', 'Cô kể chuyện, hỏi thử về cửa nhỏ.', 'hoi', 'train', 8),
    ('S0174', 'Cô kể chuyện, hỏi thử về chỗ cũ.', 'hoi', 'train', 8),
    ('S0175', 'Cô kể chuyện, hỏi thử về chỗ gửi.', 'hoi', 'train', 8),
    ('S0176', 'Mở cửa sổ, để gió thổi.', 'hoi', 'train', 6),
    ('S0177', 'Mở cửa sổ, để gió thổi nhẹ.', 'hoi', 'test', 7),
    ('S0178', 'Mở cửa sổ, để gió thổi khẽ.', 'hoi', 'train', 7),
    ('S0179', 'Mở cửa sổ, để gió đổi chiều.', 'hoi', 'train', 7),
    ('S0180', 'Mở cửa sổ, để gió thổi mạnh.', 'hoi', 'test', 7),
    ('S0181', 'Bạn gửi thử, rồi hỏi ở cửa sổ.', 'hoi', 'train', 8),
    ('S0182', 'Bạn gửi thử, rồi hỏi ở chỗ nghỉ.', 'hoi', 'train', 8),
    ('S0183', 'Bạn gửi thử, rồi hỏi ở cửa nhỏ.', 'hoi', 'train', 8),
    ('S0184', 'Bạn gửi thử, rồi hỏi ở chỗ cũ.', 'hoi', 'test', 8),
    ('S0185', 'Bạn gửi thử, rồi hỏi ở cửa cũ.', 'hoi', 'test', 8),
    ('S0186', 'Tủ gỗ ở giữa phòng, cửa sổ mở.', 'hoi', 'train', 8),
    ('S0187', 'Tủ gỗ ở giữa phòng, cửa sổ nhỏ.', 'hoi', 'train', 8),
    ('S0188', 'Tủ gỗ ở giữa phòng, cửa sổ cũ.', 'hoi', 'train', 8),
    ('S0189', 'Tủ gỗ ở giữa phòng, cửa sổ hở.', 'hoi', 'train', 8),
    ('S0190', 'Tủ gỗ ở giữa phòng, cửa sổ rộng.', 'hoi', 'train', 8),
    ('S0191', 'Cô hỏi thử, sửa cửa sổ.', 'hoi', 'train', 6),
    ('S0192', 'Cô hỏi thử, sửa cửa nhỏ.', 'hoi', 'train', 6),
    ('S0193', 'Cô hỏi thử, sửa cửa cũ.', 'hoi', 'train', 6),
    ('S0194', 'Cô hỏi thử, sửa cửa mở.', 'hoi', 'test', 6),
    ('S0195', 'Cô hỏi thử, sửa cửa hở.', 'hoi', 'train', 6),
    ('S0196', 'Để tờ giấy ở chỗ cũ, rồi mở cửa.', 'hoi', 'test', 9),
    ('S0197', 'Để tờ giấy ở chỗ nghỉ, rồi mở cửa.', 'hoi', 'train', 9),
    ('S0198', 'Để tờ giấy ở chỗ gửi, rồi mở cửa.', 'hoi', 'test', 9),
    ('S0199', 'Để tờ giấy ở chỗ sửa, rồi mở cửa.', 'hoi', 'train', 9),
    ('S0200', 'Để tờ giấy ở chỗ rửa, rồi mở cửa.', 'hoi', 'train', 9),
    ('S0201', 'Mỗi bữa, những đứa trẻ vẫn nghĩ về mẹ.', 'nga', 'train', 9),
    ('S0202', 'Mỗi bữa, những đứa trẻ vẫn nghĩ về nhà.', 'nga', 'train', 9),
    ('S0203', 'Mỗi bữa, những đứa trẻ vẫn nghĩ về lời hứa.', 'nga', 'train', 10),
    ('S0204', 'Mỗi bữa, những đứa trẻ vẫn nghĩ về chuyện cũ.', 'nga', 'train', 10),
    ('S0205', 'Mỗi bữa, những đứa trẻ vẫn nghĩ về nghĩa tình.', 'nga', 'train', 10),
    ('S0206', 'Lễ nghĩa giữ những điều đáng nhớ.', 'nga', 'test', 7),
    ('S0207', 'Lễ nghĩa giữ những điều quý giá.', 'nga', 'train', 7),
    ('S0208', 'Lễ nghĩa giữ những điều còn mãi.', 'nga', 'test', 7),
    ('S0209', 'Lễ nghĩa giữ những điều đã hứa.', 'nga', 'train', 7),
    ('S0210', 'Lễ nghĩa giữ những điều nghĩa tình.', 'nga', 'train', 7),
    ('S0211', 'Bão lũ đã làm vỡ những bức tường.', 'nga', 'train', 8),
    ('S0212', 'Bão lũ đã làm vỡ những mái ngói.', 'nga', 'train', 8),
    ('S0213', 'Bão lũ đã làm vỡ những cửa kính.', 'nga', 'train', 8),
    ('S0214', 'Bão lũ đã làm vỡ những bờ kè.', 'nga', 'train', 8),
    ('S0215', 'Bão lũ đã làm vỡ những căn nhà.', 'nga', 'train', 8),
    ('S0216', 'Mỹ mãn, vĩnh viễn, nghĩa tình còn mãi.', 'nga', 'train', 8),
    ('S0217', 'Mỹ mãn, vĩnh viễn, nghĩa tình còn mãi mãi.', 'nga', 'train', 9),
    ('S0218', 'Mỹ mãn, vĩnh viễn, nghĩa tình còn ở lại.', 'nga', 'train', 9),
    ('S0219', 'Mỹ mãn, vĩnh viễn, nghĩa tình còn trong lòng.', 'nga', 'train', 9),
    ('S0220', 'Mỹ mãn, vĩnh viễn, nghĩa tình còn nơi ấy.', 'nga', 'train', 9),
    ('S0221', 'Cũng có những người giữ lời hứa.', 'nga', 'train', 7),
    ('S0222', 'Cũng có những người giữ nghĩa tình.', 'nga', 'train', 7),
    ('S0223', 'Cũng có những người giữ kỷ niệm.', 'nga', 'train', 7),
    ('S0224', 'Cũng có những người giữ chuyện cũ.', 'nga', 'test', 7),
    ('S0225', 'Cũng có những người giữ bí mật.', 'nga', 'test', 7),
    ('S0226', 'Bữa ấy, mẹ vẫn nhớ mãi lời hứa.', 'nga', 'train', 8),
    ('S0227', 'Bữa ấy, mẹ vẫn nhớ mãi chuyện cũ.', 'nga', 'train', 8),
    ('S0228', 'Bữa ấy, mẹ vẫn nhớ mãi những ngày ấy.', 'nga', 'train', 9),
    ('S0229', 'Bữa ấy, mẹ vẫn nhớ mãi nghĩa tình.', 'nga', 'train', 8),
    ('S0230', 'Bữa ấy, mẹ vẫn nhớ mãi kỷ niệm.', 'nga', 'train', 8),
    ('S0231', 'Những kỷ niệm ấy vẫn mãi vẹn nguyên.', 'nga', 'train', 8),
    ('S0232', 'Những kỷ niệm ấy vẫn mãi vẹn tròn.', 'nga', 'train', 8),
    ('S0233', 'Những kỷ niệm ấy vẫn mãi vẹn đầy.', 'nga', 'train', 8),
    ('S0234', 'Những kỷ niệm ấy vẫn mãi vẹn tình.', 'nga', 'test', 8),
    ('S0235', 'Những kỷ niệm ấy vẫn mãi vẹn nghĩa.', 'nga', 'test', 8),
    ('S0236', 'Đã lỡ, những chuyện cũ vẫn còn mãi.', 'nga', 'train', 8),
    ('S0237', 'Đã lỡ, những chuyện cũ vẫn ở đó.', 'nga', 'train', 8),
    ('S0238', 'Đã lỡ, những chuyện cũ vẫn trở lại.', 'nga', 'train', 8),
    ('S0239', 'Đã lỡ, những chuyện cũ vẫn ám ảnh.', 'nga', 'train', 8),
    ('S0240', 'Đã lỡ, những chuyện cũ vẫn chưa quên.', 'nga', 'train', 8),
    ('S0241', 'Mỗi người giữ những kỷ niệm đã qua.', 'nga', 'train', 8),
    ('S0242', 'Mỗi người giữ những chuyện cũ đã qua.', 'nga', 'test', 8),
    ('S0243', 'Mỗi người giữ những lời hứa đã qua.', 'nga', 'train', 8),
    ('S0244', 'Mỗi người giữ những ngày cũ đã qua.', 'nga', 'test', 8),
    ('S0245', 'Mỗi người giữ những nghĩa tình đã qua.', 'nga', 'train', 8),
    ('S0246', 'Những kỷ niệm ấy vẫn mãi trong tâm trí.', 'nga', 'train', 9),
    ('S0247', 'Những kỷ niệm ấy vẫn mãi trong ký ức.', 'nga', 'test', 9),
    ('S0248', 'Những kỷ niệm ấy vẫn mãi trong lòng mình.', 'nga', 'test', 9),
    ('S0249', 'Những kỷ niệm ấy vẫn mãi trong nỗi nhớ.', 'nga', 'train', 9),
    ('S0250', 'Những kỷ niệm ấy vẫn mãi trong tâm tưởng.', 'nga', 'train', 9),
    ('S0251', 'Một bạn đặt bát thịt cạnh bếp.', 'nang', 'test', 7),
    ('S0252', 'Một bạn đặt bát thịt cạnh mâm.', 'nang', 'train', 7),
    ('S0253', 'Một bạn đặt bát thịt cạnh chạn.', 'nang', 'train', 7),
    ('S0254', 'Một bạn đặt bát thịt cạnh mặt đất.', 'nang', 'train', 8),
    ('S0255', 'Một bạn đặt bát thịt cạnh góc bếp.', 'nang', 'test', 8),
    ('S0256', 'Bạn nhặt vật thật sạch, đặt xuống đất.', 'nang', 'train', 8),
    ('S0257', 'Bạn nhặt vật thật sạch, đặt cạnh bếp.', 'nang', 'train', 8),
    ('S0258', 'Bạn nhặt vật thật sạch, đặt sát mặt đất.', 'nang', 'test', 9),
    ('S0259', 'Bạn nhặt vật thật sạch, đặt vào hộp.', 'nang', 'train', 8),
    ('S0260', 'Bạn nhặt vật thật sạch, đặt lên bàn.', 'nang', 'train', 8),
    ('S0261', 'Mặt đất ẩm, bước chân chật hẹp.', 'nang', 'train', 7),
    ('S0262', 'Mặt đất ẩm, bước chân chật chội.', 'nang', 'train', 7),
    ('S0263', 'Mặt đất ẩm, bước chân chật nặng nề.', 'nang', 'test', 8),
    ('S0264', 'Mặt đất ẩm, bước chân chật mệt mỏi.', 'nang', 'train', 8),
    ('S0265', 'Mặt đất ẩm, bước chân chật chật chội.', 'nang', 'train', 8),
    ('S0266', 'Đặt tập sách thật gọn, đừng để lệch.', 'nang', 'train', 8),
    ('S0267', 'Đặt tập sách thật gọn, đừng làm rách.', 'nang', 'train', 8),
    ('S0268', 'Đặt tập sách thật gọn, đừng đặt chồng.', 'nang', 'train', 8),
    ('S0269', 'Đặt tập sách thật gọn, đừng bỏ sót.', 'nang', 'train', 8),
    ('S0270', 'Đặt tập sách thật gọn, đừng quên mất.', 'nang', 'train', 8),
    ('S0271', 'Một chiếc bát đặt cạnh bếp.', 'nang', 'train', 6),
    ('S0272', 'Một chiếc bát đặt cạnh chạn.', 'nang', 'train', 6),
    ('S0273', 'Một chiếc bát đặt cạnh mâm cơm.', 'nang', 'train', 7),
    ('S0274', 'Một chiếc bát đặt cạnh mặt đất.', 'nang', 'train', 7),
    ('S0275', 'Một chiếc bát đặt cạnh bậc thềm.', 'nang', 'train', 7),
    ('S0276', 'Bạn đọc chậm, viết thật đẹp.', 'nang', 'test', 6),
    ('S0277', 'Bạn đọc chậm, viết thật cẩn thận.', 'nang', 'test', 7),
    ('S0278', 'Bạn đọc chậm, viết thật ngắn gọn.', 'nang', 'train', 7),
    ('S0279', 'Bạn đọc chậm, viết thật đúng mực.', 'nang', 'train', 7),
    ('S0280', 'Bạn đọc chậm, viết thật rõ ràng.', 'nang', 'train', 7),
    ('S0281', 'Mẹ nhặt hạt gạo trên mặt đất.', 'nang', 'train', 7),
    ('S0282', 'Mẹ nhặt hạt gạo trên mặt bàn.', 'nang', 'train', 7),
    ('S0283', 'Mẹ nhặt hạt gạo trên mặt thềm.', 'nang', 'train', 7),
    ('S0284', 'Mẹ nhặt hạt gạo trên mặt mâm.', 'nang', 'train', 7),
    ('S0285', 'Mẹ nhặt hạt gạo trên mặt chạn.', 'nang', 'train', 7),
    ('S0286', 'Cột chặt túi gạo, đặt sát bếp.', 'nang', 'train', 7),
    ('S0287', 'Cột chặt túi gạo, đặt sát mặt đất.', 'nang', 'train', 8),
    ('S0288', 'Cột chặt túi gạo, đặt sát vách.', 'nang', 'train', 7),
    ('S0289', 'Cột chặt túi gạo, đặt sát cạnh bàn.', 'nang', 'train', 8),
    ('S0290', 'Cột chặt túi gạo, đặt sát góc bếp.', 'nang', 'test', 8),
    ('S0291', 'Đặt bút mực cạnh quyển tập.', 'nang', 'train', 6),
    ('S0292', 'Đặt bút mực cạnh quyển vở.', 'nang', 'train', 6),
    ('S0293', 'Đặt bút mực cạnh quyển sách.', 'nang', 'train', 6),
    ('S0294', 'Đặt bút mực cạnh quyển nhật ký.', 'nang', 'test', 7),
    ('S0295', 'Đặt bút mực cạnh quyển sổ tay.', 'nang', 'train', 7),
    ('S0296', 'Thật mệt, bạn đặt vật xuống đất.', 'nang', 'test', 7),
    ('S0297', 'Thật mệt, bạn đặt vật xuống mặt bàn.', 'nang', 'test', 8),
    ('S0298', 'Thật mệt, bạn đặt vật xuống cạnh bếp.', 'nang', 'train', 8),
    ('S0299', 'Thật mệt, bạn đặt vật xuống bậc thềm.', 'nang', 'train', 8),
    ('S0300', 'Thật mệt, bạn đặt vật xuống nền gạch.', 'nang', 'train', 8),
]

from collections import Counter
CAU = CAU_CHUAN[:SO_CAU] if not CHAY_THU else CAU_CHUAN[:3]
print(f"Câu sẽ sinh: {len(CAU)}")
print("  theo thanh:", dict(Counter(c[2] for c in CAU)))
print("  theo tập  :", dict(Counter(c[3] for c in CAU)))
print("  ví dụ     :", CAU[0][1])
if CHAY_THU:
    print("\n⚠️ CHẠY THỬ — 3 câu. Đặt CHAY_THU = False ở ô 1 rồi chạy lại.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 3 · CÀI GÓI
# ═══════════════════════════════════════════════════════════════════════════
def cai(ten, import_ten=None, bat_buoc=True):
    import_ten = import_ten or ten.replace("-", "_")
    if importlib.util.find_spec(import_ten) is not None:
        print(f"  ✓ {ten:12s} có sẵn"); return True
    print(f"  · {ten:12s} chưa có → pip install")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", ten],
                       capture_output=True, text=True)
    importlib.invalidate_caches()
    ok = importlib.util.find_spec(import_ten) is not None
    print("      ✓ cài xong" if ok else
          "      ✗ CÀI HỎNG: " + ((r.stderr or r.stdout or "")[-250:].replace("\n", " ") or "không rõ"))
    if not ok and bat_buoc:
        raise RuntimeError(f"Không cài được `{ten}`. Internet đã bật chưa?")
    return ok

cai("soundfile")
cai("librosa")
cai("vieneu")
cai("omnivoice")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 4 · 7 TỆP .pt → VoiceClonePrompt
# ═══════════════════════════════════════════════════════════════════════════
def tim_giong():
    co = {}
    for g in GIONG:
        p = os.path.join(GOC_GIONG, g + ".pt")
        if os.path.isfile(p):
            co[g] = p; continue
        for cap in range(1, 5):
            hit = glob.glob("/kaggle/input/" + "*/" * cap + g + ".pt")
            if hit:
                co[g] = hit[0]; break
    return co

TIM_GIONG = tim_giong()
print(f"Tìm thấy {len(TIM_GIONG)}/{len(GIONG)} tệp .pt")
for g in GIONG:
    print("   ", "✓" if g in TIM_GIONG else "✗", f"{g:16s}", TIM_GIONG.get(g, "KHÔNG THẤY"))
if len(TIM_GIONG) < len(GIONG):
    raise FileNotFoundError("Thiếu .pt — Add Input dataset `tts-pro-vip`.")

VC = None
for mod in ("omnivoice", "omnivoice.prompt", "omnivoice.voice_clone",
            "omnivoice.types", "omnivoice.utils", "omnivoice.inference"):
    try:
        _m = importlib.import_module(mod)
        if getattr(_m, "VoiceClonePrompt", None) is not None:
            VC = getattr(_m, "VoiceClonePrompt"); print(f"VoiceClonePrompt ← {mod}"); break
    except Exception:
        pass
if VC is None:
    raise RuntimeError("Không thấy lớp VoiceClonePrompt trong gói omnivoice.")

PROMPT = {g: VC.load(p) for g, p in TIM_GIONG.items()}
print("Đọc xong:", {g: type(v).__name__ for g, v in PROMPT.items()})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 5 · NẠP OmniVoice GỐC (+ OmDiOm-ft) VÀ DÒ CHỮ KÝ generate()
# ═══════════════════════════════════════════════════════════════════════════
import contextlib

def inference_mode():
    return torch.inference_mode() if hasattr(torch, "inference_mode") else contextlib.nullcontext()

def chuan_hoa_am(a, sr_mac_dinh):
    """Đưa bất kỳ kiểu trả về nào (tensor/dict/tuple/ndarray) về (mảng 1-D float32, sr)."""
    if isinstance(a, tuple) and len(a) == 2 and isinstance(a[1], (int, float)):
        x, _ = chuan_hoa_am(a[0], int(a[1])); return x, int(a[1])
    if isinstance(a, dict):
        for k in ("audio", "wav", "waveform", "data", "arr", "y"):
            if k in a: return chuan_hoa_am(a[k], sr_mac_dinh)
    for k in ("audio", "wav", "waveform", "data"):
        if hasattr(a, k): return chuan_hoa_am(getattr(a, k), sr_mac_dinh)
    if hasattr(a, "detach"): a = a.detach().float().cpu().numpy()
    x = np.asarray(a, dtype=np.float32)
    if x.ndim > 1:
        x = x.mean(axis=0) if x.shape[0] <= 8 else x.mean(axis=1)
    return np.ascontiguousarray(x.reshape(-1)), int(sr_mac_dinh)

co_cuda = torch.cuda.is_available()

def nap_omni(p, nhan="OmniVoice"):
    from omnivoice import OmniVoice
    loi = []
    for cach, kw in (("from_pretrained+dev",
                      dict(device_map="cuda" if co_cuda else "cpu",
                           dtype=torch.float16 if co_cuda else torch.float32)),
                     ("from_pretrained", {})):
        try:
            m = OmniVoice.from_pretrained(p, **kw)
            print(f"    ✓ {nhan} nạp bằng .{cach}")
            return m
        except Exception as e:
            loi.append(f"{cach}: {type(e).__name__}: {str(e)[:90]}")
    raise RuntimeError(f"Không nạp được {nhan}:\n  " + "\n  ".join(loi))

OMNI = nap_omni("k2-fsa/OmniVoice", "OmniVoice gốc")
SR_OV = int(getattr(OMNI, "sampling_rate", 0) or 24000)
print(f"✓ OmniVoice gốc · sr = {SR_OV}")

# dò chữ ký generate() một lần — các bản cài khác nhau có thể thiếu num_step
def _thu_ov(m, kw):
    try:
        with inference_mode():
            o = m.generate(text="Xin chào.", language="vi",
                           voice_clone_prompt=PROMPT[GIONG[0]], **kw)
        x, _ = chuan_hoa_am(o, SR_OV)
        return x.size > 0
    except Exception:
        return False

OV_KW = None
for kw in ({"num_step": NUM_STEP, "postprocess_output": True},
           {"num_step": NUM_STEP}, {"postprocess_output": True}, {}):
    if _thu_ov(OMNI, kw):
        OV_KW = kw; break
if OV_KW is None:
    raise RuntimeError("OmniVoice.generate() không gọi được với bất kỳ chữ ký nào.")
print(f"✓ chữ ký OmniVoice: generate(**{list(OV_KW) or 'mặc định'})")

# ── OmDiOm fine-tune 4.000 bước ────────────────────────────────────────────
def tim_trong_so_ft():
    if DUONG_FT:
        return DUONG_FT
    ung = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if glob.glob(os.path.join(d, "*.safetensors")) or glob.glob(os.path.join(d, "*.bin")):
            ung.append(d)
    return ung

OMDIOM = None
ft = tim_trong_so_ft()
if isinstance(ft, str):
    try:
        OMDIOM = nap_omni(ft, "OmDiOm-ft"); print(f"✓ OmDiOm-ft · {ft}")
    except Exception as e:
        print(f"!! Không nạp được OmDiOm-ft: {e}")
elif len(ft) == 1:
    try:
        OMDIOM = nap_omni(ft[0], "OmDiOm-ft"); print(f"✓ OmDiOm-ft (tự dò) · {ft[0]}")
    except Exception as e:
        print(f"!! Không nạp được OmDiOm-ft: {e}")
elif len(ft) > 1:
    print("⚠️ Tìm thấy NHIỀU thư mục trọng số — không đoán bừa, bỏ qua OmDiOm-ft:")
    for d in ft: print("     ", d)
    print("   → Điền DUONG_FT đúng ở ô 1 rồi chạy lại ô 5.")
else:
    print("!! KHÔNG TÌM THẤY trọng số OmDiOm-ft trong /kaggle/input")
    print("   → Add Input dataset chứa trọng số fine-tune, hoặc điền DUONG_FT ở ô 1.")
    print("   → Vẫn chạy tiếp với 2 bộ sinh; chạy bù OmDiOm-ft sau được (có resume).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 6 · REF CHO 7 GIỌNG — OmniVoice đọc 1 câu dài bằng prompt .pt → wav 6–8 s
# ═══════════════════════════════════════════════════════════════════════════
# VieNeu cần `ref_audio` là ĐƯỜNG DẪN WAV 3–8 s; tệp .pt là prompt của OmniVoice.
# Nên phải "dịch" .pt → wav một lần cho mỗi giọng.  (Bài học 30/10: ref lấy từ câu
# ngắn 1–2 s thì 0/7 giọng qua cổng 3 s ⇒ 0 clip. Ở đây dùng câu riêng, dài.)
import soundfile as sf

def cat_lang(x, sr, nguong_db=-45.0):
    hop = max(1, int(0.02 * sr)); n = len(x) // hop
    if n < 3: return x
    e = 20*np.log10(np.sqrt(np.mean(x[:n*hop].reshape(n, hop)**2, axis=1)) + 1e-9)
    giu = np.where(e > nguong_db)[0]
    if len(giu) == 0: return x
    a = max(0, giu[0]-2); b = min(n, giu[-1]+3)
    return x[a*hop:b*hop]

REF = {}
for g in GIONG:
    p = os.path.join(DIR_REF, g + ".wav")
    if os.path.isfile(p):
        x, sr = sf.read(p, dtype="float32"); REF[g] = p
        print(f"  {g:16s} dùng lại ref {len(x)/sr:5.2f} s"); continue
    out = OMNI.generate(text=CAU_REF, language="vi", voice_clone_prompt=PROMPT[g], **OV_KW)
    x, sr = chuan_hoa_am(out, SR_OV)
    x = cat_lang(x, sr)
    if len(x)/sr > 9.0:
        x = cat_lang(x[:int(8.5*sr)], sr)
    sf.write(p, x, sr)
    REF[g] = p
    canh = "  ⚠️ < 3 s, VieNeu có thể từ chối" if len(x)/sr < 3.0 else ""
    print(f"  {g:16s} ref {len(x)/sr:5.2f} s · {sr} Hz{canh}")
print(f"\nRef xong: {len(REF)}/7")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 7 · NẠP VieNeu (+ dò chữ ký infer)
# ═══════════════════════════════════════════════════════════════════════════
from vieneu import Vieneu

VIENEU = None
for ten_cach, f in (("Vieneu()", lambda: Vieneu()),
                    ("Vieneu(model)", lambda: Vieneu("pnnbao-ump/VieNeu-TTS-v3-Turbo"))):
    try:
        VIENEU = f(); print(f"    ✓ dựng bằng {ten_cach}"); break
    except Exception as e:
        print(f"    · {ten_cach} hỏng: {type(e).__name__}: {str(e)[:70]}")
if VIENEU is None:
    raise RuntimeError("Không dựng được VieNeu.")
SR_VN = int(getattr(VIENEU, "sample_rate", 0) or 48000)
print(f"✓ VieNeu · sr = {SR_VN}")

def _thu_vn(kw):
    try:
        with inference_mode():
            o = VIENEU.infer(text="Xin chào.", **kw)
        x, _ = chuan_hoa_am(o, SR_VN)
        return x.size > 0
    except Exception:
        return False

VN_KW = None
for kw in ({"ref_audio": REF[GIONG[0]]}, {"ref": REF[GIONG[0]]},
           {"ref_audio_path": REF[GIONG[0]]}, {}):
    if _thu_vn(kw):
        VN_KW = kw; break
if VN_KW is None:
    raise RuntimeError("VieNeu.infer() không gọi được với bất kỳ chữ ký nào. "
                       "Gửi tôi đoạn lỗi để tôi chỉnh đúng.")
print(f"✓ chữ ký VieNeu: infer(**{list(VN_KW) or 'chỉ text'})")

BO_SINH = ["vieneu", "omnivoice"] + (["omdiom"] if OMDIOM is not None else [])
print("\nBộ sinh sẽ chạy:", BO_SINH)
if "omdiom" not in BO_SINH:
    print("⚠️ THIẾU OmDiOm-ft ⇒ chỉ còn 2 lớp. Bài T-B (3 lớp) chưa đủ.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 8 · VÒNG SINH — resume, tiến độ, tự dừng trước khi Kaggle cắt phiên
# ═══════════════════════════════════════════════════════════════════════════
import soundfile as sf

viec = [(b, g, c) for b in BO_SINH for g in GIONG for c in CAU]
print(f"Tổng việc: {len(viec)} clip "
      f"({len(BO_SINH)} bộ sinh × {len(GIONG)} giọng × {len(CAU)} câu)")

def duong_dan(g, c, b):
    d = os.path.join(DIR_AM, b, g); os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{c[0]}.{DINH_DANG}")

def sinh_mot(b):
    if b[0] == "vieneu":
        kw = {k: REF[b[1]] for k in VN_KW if VN_KW[k] is not None} if VN_KW else {}
        return chuan_hoa_am(VIENEU.infer(text=b[2][1], **kw), SR_VN)
    m = OMNI if b[0] == "omnivoice" else OMDIOM
    out = m.generate(text=b[2][1], language="vi",
                     voice_clone_prompt=PROMPT[b[1]], **OV_KW)
    return chuan_hoa_am(out, SR_OV)

da_co = {p for b in BO_SINH for p in glob.glob(os.path.join(DIR_AM, b, "*", f"*.{DINH_DANG}"))}
print(f"Đã có sẵn từ lần trước: {len(da_co)} clip")

t0 = time.time(); lam = 0; loi = []; dung_som = False
for i, b in enumerate(viec):
    p = duong_dan(b[1], b[2], b[0])
    if os.path.isfile(p):
        continue
    if time.time() - t0 > GIO_TOI_DA * 3600:
        dung_som = True; break
    try:
        torch.manual_seed(SEED + i)
        with inference_mode():
            x, sr = sinh_mot(b)
        if x.size < 0.2 * sr:
            raise ValueError(f"quá ngắn ({x.size/sr:.2f} s)")
        sf.write(p, x, sr)
        lam += 1
    except Exception as e:
        loi.append([b[0], b[1], b[2][0], f"{type(e).__name__}: {str(e)[:90]}"])
    if lam and lam % 25 == 0:
        troi = time.time() - t0
        con = len(viec) - i - 1
        print(f"  {lam:5d} clip · {troi/60:5.1f} phút · "
              f"còn ~{con*troi/max(lam,1)/60:5.1f} phút · lỗi {len(loi)}", flush=True)

print(f"\nXong: +{lam} clip trong {(time.time()-t0)/60:.1f} phút · lỗi {len(loi)}")
if dung_som:
    print("⏰ Chạm giới hạn giờ. Chạy lại ô 8 (cùng output) để làm tiếp — có resume.")
for l in loi[:15]:
    print("   ✗", l)
if loi:
    json.dump(loi, open(os.path.join(RA, "loi_sinh.json"), "w"), ensure_ascii=False, indent=1)
    print(f"   (đã ghi {len(loi)} lỗi vào loi_sinh.json)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 9 · MANIFEST + KIỂM TRA ĐỦ / THIẾU
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd

dong = []
for b in BO_SINH:
    for g in GIONG:
        for c in CAU:
            p = duong_dan(g, c, b)
            if not os.path.isfile(p):
                continue
            try:
                info = sf.info(p)
                dong.append(dict(duong_dan=p, cau_id=c[0], giong=g, bo_sinh=b,
                                 sr=info.samplerate, giay=info.frames/info.samplerate,
                                 so_am_tiet=c[4], la_ref=0))
            except Exception:
                pass

MF = pd.DataFrame(dong)
MF.to_csv(os.path.join(RA, "manifest.csv"), index=False, encoding="utf-8-sig")

print("=" * 72)
print(f"MANIFEST: {len(MF)} dòng")
if len(MF):
    print(pd.crosstab(MF.bo_sinh, MF.giong, margins=True, margins_name="TỔNG").to_string())
    print(f"\nTổng {MF.giay.sum()/60:.1f} phút · TB {MF.giay.mean():.2f} s/clip "
          f"· sr {sorted(int(x) for x in MF.sr.unique())}")
    print("\nGiây theo bộ sinh:")
    print(MF.groupby("bo_sinh").giay.agg(["size", "sum", "mean"]).round(2).to_string())

can = len(BO_SINH) * len(GIONG) * len(CAU)
print("=" * 72)
print(f"Cần {can} · có {len(MF)} · thiếu {can - len(MF)}")
if len(MF) == can:
    print("✅ ĐỦ — sang ô 10 để đóng gói")
else:
    for b in BO_SINH:
        nb_ = int((MF.bo_sinh == b).sum())
        if nb_ < len(GIONG)*len(CAU):
            print(f"   ⚠️ {b}: {nb_}/{len(GIONG)*len(CAU)} — chạy lại ô 8 (resume làm tiếp)")

## Phân hệ Trực quan hóa Dữ liệu (EDA) và Phân tích Đặc trưng Phổ Âm học
Mô-đun xây dựng 17 đồ thị và bảng phân tích chuyên sâu về hình thái phổ âm thanh, phân bố nhịp điệu thời lượng, phân tích thành phần chính (PCA) và trình phát đối chứng âm thanh ghép cặp đa phương tiện.


In [ ]:
%%writefile bao_cao_html.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
bao_cao_html.py — sinh MỘT tệp HTML tự chứa: EDA + NGHE & SO SÁNH.

    tao_bao_cao("manifest.csv", "BAO_CAO.html")
    tao_bao_cao("manifest.csv", "BAO_CAO.html", so_cau_nghe=4, so_anh_moi_bo=8)

Tự chứa = ảnh và audio đều nhúng base64 ⇒ mở offline được, Ctrl+P ra PDF, chèn slide được.
Không cần mạng, không cần thư mục kèm.
"""
from __future__ import annotations

import base64
import io
import os
from datetime import datetime

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

CAN = ["duong_dan", "cau_id", "giong", "bo_sinh", "sr", "giay"]
THU_TU_THANH = ["ngang", "huyen", "sac", "hoi", "nga", "nang"]

MAU = {"vieneu": "#2E86AB", "omnivoice": "#E4572E", "omdiom": "#17A398", "that": "#6A4C93"}
MAU_DU = ["#2E86AB", "#E4572E", "#17A398", "#6A4C93", "#F4A259", "#5B8E7D", "#BC4B51"]
MIME = {".flac": "audio/flac", ".wav": "audio/wav", ".mp3": "audio/mpeg",
        ".m4a": "audio/mp4", ".ogg": "audio/ogg", ".opus": "audio/ogg"}

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "-",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10.5, "axes.titlesize": 12.5, "axes.titleweight": "600",
    "axes.labelsize": 10.5, "figure.autolayout": True,
    "savefig.facecolor": "white",
})

_HINH: list[dict] = []


# ═══════════════════════════════════════════════════════════════ tiện ích
def _mau(bo):
    return MAU.get(bo, "#7A8794")


def _md(t):
    """Đổi **đậm** trong chú thích thành <b>đậm</b> — chú thích viết theo lối markdown cho dễ đọc."""
    import re as _re
    t = _re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", str(t))
    return _re.sub(r"(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)", r"<i>\1</i>", t)


def _anh(fig, tieu_de, doc_ra, rong="100%"):
    tieu_de = tieu_de.rstrip(" .")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=140, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    _HINH.append({"tieu_de": tieu_de, "doc_ra": _md(doc_ra),
                  "b64": base64.b64encode(buf.getvalue()).decode(), "rong": rong})
    print(f"   ✓ {tieu_de}")


def _bang(df, tieu_de, ghi_chu=""):
    tieu_de = tieu_de.rstrip(" .")
    _HINH.append({"tieu_de": tieu_de, "doc_ra": _md(ghi_chu),
                  "html": df.to_html(index=False, border=0, classes="bang", escape=False)})
    print(f"   ✓ {tieu_de} (bảng)")


def _doc(p):
    """Đọc audio. soundfile trước — nhanh và không kéo theo torch; librosa là dự phòng."""
    try:
        import soundfile as sf
        y, sr = sf.read(p, dtype="float32", always_2d=False)
        if getattr(y, "ndim", 1) > 1:
            y = y.mean(axis=1)
        if len(y):
            return np.asarray(y, dtype=np.float32), int(sr)
    except Exception:
        pass
    import librosa
    return librosa.load(p, sr=None, mono=True)


def _nhung_bytes(p):
    with open(p, "rb") as f:
        b = f.read()
    mime = MIME.get(os.path.splitext(p)[1].lower(), "audio/wav")
    return "data:%s;base64,%s" % (mime, base64.b64encode(b).decode())


def _do_clip(y, sr):
    """Ba con số đọc kèm khi nghe: F0 trung vị, rolloff95, % năng lượng băng cao."""
    import librosa
    if sr != 16000:
        y16 = librosa.resample(y, orig_sr=sr, target_sr=16000)
        sr16 = 16000
    else:
        y16, sr16 = y, sr
    S = np.abs(librosa.stft(y16, n_fft=1024, hop_length=160))
    freqs = librosa.fft_frequencies(sr=sr16, n_fft=1024)
    tong = float(np.sum(S ** 2) + 1e-12)
    hf = 100 * float(np.sum(S[freqs > sr16 * 0.25] ** 2)) / tong
    roll = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=sr16, roll_percent=0.95)))
    try:
        f0 = librosa.yin(y16, fmin=60, fmax=400, sr=sr16, frame_length=1024, hop_length=160)
        f0 = f0[np.isfinite(f0) & (f0 > 60) & (f0 < 400)]
        f0m = float(np.median(f0)) if f0.size > 8 else float("nan")
    except Exception:
        f0m = float("nan")
    return f0m, roll, hf


# ═══════════════════════════════════════════════════════ 1. BỘ CÂU
def ve_bo_cau(b):
    if b is None or "thanh" not in b.columns:
        return

    fig, ax = plt.subplots(figsize=(9, 3.4))
    dem = b.thanh.value_counts().reindex(THU_TU_THANH).fillna(0)
    bars = ax.bar(dem.index, dem.values, color=MAU_DU[:6], edgecolor="white", linewidth=1.6)
    for r, v in zip(bars, dem.values):
        ax.text(r.get_x() + r.get_width() / 2, v + dem.max() * 0.02, int(v),
                ha="center", fontsize=11.5, fontweight="700")
    ax.set_title("Số câu theo thanh điệu")
    ax.set_ylabel("Số câu"); ax.set_ylim(0, dem.max() * 1.2)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    _anh(fig, "300 câu · 50 câu mỗi thanh",
         "Cân bằng thanh điệu để không thanh nào chiếm ưu thế trong tập đánh giá — nếu lệch, "
         "mô hình có thể đạt điểm cao chỉ nhờ đoán thanh phổ biến nhất.")

    if "so_am_tiet" in b.columns:
        fig, ax = plt.subplots(figsize=(9, 3.3))
        vc = b.so_am_tiet.value_counts().sort_index()
        ax.bar(vc.index, vc.values, color="#6A4C93", alpha=0.88, edgecolor="white")
        m = b.so_am_tiet.mean()
        ax.axvline(m, color="#E4572E", ls="--", lw=2, label=f"trung bình {m:.2f} âm tiết")
        ax.set_title("Độ dài câu"); ax.set_xlabel("Số âm tiết"); ax.set_ylabel("Số câu")
        ax.legend(frameon=False)
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        _anh(fig, "Độ dài câu: 5–10 âm tiết",
             f"Trung bình **{m:.2f} âm tiết/câu** — đủ ngắn để sinh nhanh, đủ dài để MFCC bắt được âm sắc.")

    if "tap" in b.columns:
        fig, ax = plt.subplots(figsize=(6.4, 3.3))
        ct = pd.crosstab(b.thanh, b.tap).reindex(THU_TU_THANH)
        ct.plot(kind="bar", ax=ax, color=["#2E86AB", "#E4572E"], width=0.8,
                edgecolor="white", linewidth=1.2)
        ax.set_title("Chia tập theo câu"); ax.set_ylabel("Số câu"); ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(title="", frameon=False)
        _anh(fig, "240 train / 60 test — mỗi thanh 40/10",
             "Câu ở test **không** xuất hiện ở train. Chia ngẫu nhiên theo clip là sai: cùng một câu "
             "nằm cả hai phía thì mô hình chỉ cần học thuộc câu, không học giọng.", rong="70%")


# ═══════════════════════════════════════════════════ 2. ĐỘ ĐẦY ĐỦ
def ve_day_du(d):
    pt = d.pivot_table(index="bo_sinh", columns="giong", values="giay",
                       aggfunc="size", fill_value=0)
    fig, ax = plt.subplots(figsize=(10, max(2.3, 0.6 * len(pt) + 1.5)))
    im = ax.imshow(pt.values, cmap="YlGnBu", aspect="auto")
    ax.set_xticks(range(len(pt.columns)), pt.columns, rotation=28, ha="right")
    ax.set_yticks(range(len(pt.index)), pt.index)
    vmax = pt.values.max() if pt.values.size else 1
    for i in range(pt.shape[0]):
        for j in range(pt.shape[1]):
            v = pt.values[i, j]
            vmin = pt.values.min()
            frac = (v - vmin) / (vmax - vmin) if vmax > vmin else 0.0   # 0..1 theo thang màu
            ax.text(j, i, int(v), ha="center", va="center", fontsize=10.5, fontweight="700",
                    color="white" if frac > 0.55 else "#1b1b1b")
    ax.set_title("Số clip theo (bộ sinh × giọng)")
    ax.grid(False)
    if im.get_clim()[0] != im.get_clim()[1]:
        fig.colorbar(im, ax=ax, shrink=0.72, label="số clip")
    else:
        ax.set_title("Số clip theo (bộ sinh × giọng) — lưới đầy đủ ở mọi ô")
    _anh(fig, "Độ đầy đủ lưới bộ sinh × giọng",
         "Ô lệch số nghĩa là bộ sinh đó lỗi ở giọng tương ứng. Lỗi loại này hay xảy ra khi chạy dài "
         "và Kaggle cắt phiên — nhìn lưới là thấy ngay, không phải dò log.")

    t = d.pivot_table(index="bo_sinh", columns="giong", values="giay", aggfunc="size",
                      fill_value=0).reset_index()
    _bang(t, "Bảng đếm clip", "Mỗi ô phải bằng đúng số câu đã sinh cho giọng đó.")


# ═══════════════════════════════════════════════ 3. THỜI LƯỢNG & NHỊP
def ve_thoi_luong(d):
    bo = sorted(d.bo_sinh.unique())

    fig, ax = plt.subplots(figsize=(9.5, 4))
    du = [d[d.bo_sinh == b].giay.values for b in bo]
    ok = [v for v in du if len(v) > 2]
    if len(ok) == len(du):
        parts = ax.violinplot(du, showmeans=True, showextrema=False)
        for pc, b in zip(parts["bodies"], bo):
            pc.set_facecolor(_mau(b)); pc.set_alpha(0.62); pc.set_edgecolor("white")
    rng = np.random.default_rng(0)
    for i, (b, v) in enumerate(zip(bo, du), start=1):
        ax.scatter(np.full(len(v), i) + rng.uniform(-0.075, 0.075, len(v)),
                   v, s=13, color="#20262d", alpha=0.42, zorder=3)
        ax.text(i, np.max(v) + 0.055, f"TB {np.mean(v):.2f}s", ha="center",
                fontsize=10.5, fontweight="700", color=_mau(b))
    ax.set_xticks(range(1, len(bo) + 1), bo)
    ax.set_ylabel("Thời lượng (giây)")
    ax.set_title("Phân bố thời lượng theo bộ sinh")
    _anh(fig, "Thời lượng mỗi clip theo bộ sinh",
         "Ba bộ sinh phải cho phân bố **gần nhau**. Nếu một bộ dài vượt trội, thời lượng trở thành "
         "đặc trưng phân biệt — mô hình học nó thay vì học âm sắc, và kết quả mất ý nghĩa.")

    fig, ax = plt.subplots(figsize=(11, 3.8))
    pt = d.pivot_table(index="giong", columns="bo_sinh", values="giay", aggfunc="mean")
    pt = pt.reindex(sorted(pt.index))
    x = np.arange(len(pt)); w = 0.8 / max(len(bo), 1)
    for k, b in enumerate(bo):
        if b in pt.columns:
            ax.bar(x + k * w - 0.4 + w / 2, pt[b].values, w * 0.92, label=b,
                   color=_mau(b), edgecolor="white", linewidth=1)
    ax.set_xticks(x, pt.index, rotation=18, ha="right")
    ax.set_ylabel("Thời lượng TB (giây)")
    ax.set_title("Thời lượng trung bình theo giọng")
    ax.legend(frameon=False, ncol=len(bo))
    _anh(fig, "Bảy giọng đọc với nhịp khác nhau",
         "Chênh lệch giữa các giọng là **tín hiệu tốt**: chứng tỏ giọng được nhân bản khác nhau thật, "
         "không phải cùng một giọng đọc lại.")

    # heatmap thời lượng TB: giọng × bộ sinh
    pt2 = d.pivot_table(index="giong", columns="bo_sinh", values="giay", aggfunc="mean")
    if pt2.shape[0] > 1 and pt2.shape[1] > 1:
        fig, ax = plt.subplots(figsize=(7.2, max(2.6, 0.46 * len(pt2) + 1.2)))
        im = ax.imshow(pt2.values, cmap="RdYlBu_r", aspect="auto")
        ax.set_xticks(range(pt2.shape[1]), pt2.columns, rotation=14)
        ax.set_yticks(range(pt2.shape[0]), pt2.index)
        for i in range(pt2.shape[0]):
            for j in range(pt2.shape[1]):
                ax.text(j, i, f"{pt2.values[i, j]:.2f}", ha="center", va="center",
                        fontsize=10.5, fontweight="700",
                        color="white" if pt2.values[i, j] > np.nanmean(pt2.values) else "#1b1b1b")
        ax.set_title("Thời lượng trung bình (giây) — giọng × bộ sinh")
        ax.grid(False)
        fig.colorbar(im, ax=ax, shrink=0.8, label="giây")
        _anh(fig, "Ma trận thời lượng theo giọng và bộ sinh",
             "Mỗi **cột** gần như thẳng hàng nghĩa là ba bộ sinh đọc cùng một câu với nhịp giống nhau — "
             "dấu hiệu chúng chia sẻ cùng kiến trúc. Cột nào lệch nhiều là bộ sinh đó đang đọc khác nhịp.")

    # bảng thống kê mô tả
    st = d.groupby("bo_sinh").giay.agg(["count", "mean", "std", "min", "median", "max"]).round(3)
    st.columns = ["số clip", "TB", "độ lệch chuẩn", "nhỏ nhất", "trung vị", "lớn nhất"]
    st = st.reset_index()
    _bang(st, "Thống kê mô tả thời lượng",
          "Độ lệch chuẩn nhỏ nghĩa là bộ sinh đọc rất đều — đặc điểm của TTS. "
          "Người thật thường có độ lệch lớn hơn.")

    if d.so_am_tiet.notna().any():
        dd = d.assign(am_giay=d.so_am_tiet / d.giay)
        fig, ax = plt.subplots(figsize=(9, 3.4))
        bins = np.linspace(dd.am_giay.min(), dd.am_giay.max(), min(16, max(5, len(dd) // 3)))
        for b in bo:
            v = dd[dd.bo_sinh == b].am_giay.values
            ax.hist(v, bins=bins, alpha=0.5, color=_mau(b), label=f"{b} (TB {np.mean(v):.2f})",
                    edgecolor="white")
        ax.set_xlabel("Tốc độ đọc (âm tiết/giây)"); ax.set_ylabel("Số clip")
        ax.set_title("Tốc độ đọc")
        ax.legend(frameon=False)
        _anh(fig, "Tốc độ đọc — chỉ số nhịp điệu",
             "TTS đọc **đều như máy**, người thật không. Đây là cơ sở của họ đặc trưng F3 "
             "(nhịp & ngắt) trong thiết kế.")


# ═══════════════════════════════════════════════ 4. TẦN SỐ LẤY MẪU
def ve_tan_so(d):
    bo = sorted(d.bo_sinh.unique())
    sr_map = {b: int(d[d.bo_sinh == b].sr.mode().iloc[0]) for b in bo}
    fig, ax = plt.subplots(figsize=(8, 2.7))
    for i, b in enumerate(bo):
        v = sr_map[b]
        ax.barh(i, v, color=_mau(b), height=0.58)
        ax.text(v + 700, i, f"{v:,} Hz", va="center", fontsize=11, fontweight="700")
    ax.set_yticks(range(len(bo)), bo)
    ax.set_xlim(0, max(sr_map.values()) * 1.22)
    ax.set_xlabel("Tần số lấy mẫu (Hz)")
    ax.set_title("Tần số lấy mẫu của từng bộ sinh")
    _anh(fig, "Ba bộ sinh KHÁC tần số lấy mẫu",
         "Đây là **cái bẫy lớn nhất của dự án**. Nếu không resample về cùng một mức, bộ phân lớp chỉ "
         "cần nhìn băng thông là tách đúng 100 % — điểm cao nhưng vô nghĩa. Thí nghiệm E4 đo đúng "
         "phần điểm ảo này.")

    t = pd.DataFrame({"bo_sinh": bo, "sr": [sr_map[b] for b in bo],
                      "số_clip": [int((d.bo_sinh == b).sum()) for b in bo]})
    _bang(t, "Tần số lấy mẫu theo bộ sinh",
          "VieNeu 48 kHz, OmniVoice và OmDiOm 24 kHz ⇒ bắt buộc resample trước khi trích đặc trưng.")


# ═══════════════════════════════════════════ 5. SO SÁNH CÙNG CÂU
def ve_cung_cau(d):
    goc = d.groupby(["giong", "cau_id"]).bo_sinh.nunique()
    du = goc[goc == d.bo_sinh.nunique()]
    if du.empty:
        return
    giong, cau = du.index[len(du) // 2]
    sel = d[(d.giong == giong) & (d.cau_id == cau)]
    bo = sorted(sel.bo_sinh.unique())

    import librosa
    fig, axes = plt.subplots(len(bo), 2, figsize=(13.5, 2.5 * len(bo)),
                             gridspec_kw={"width_ratios": [1.3, 1]})
    axes = np.atleast_2d(axes)
    for i, b in enumerate(bo):
        p = sel[sel.bo_sinh == b].iloc[0].duong_dan
        if not os.path.isfile(p):
            continue
        try:
            y, sr = librosa.load(p, sr=16000, mono=True)
        except Exception:
            continue
        t = np.arange(len(y)) / 16000
        ax = axes[i, 0]
        ax.plot(t, y, lw=0.5, color=_mau(b))
        ax.set_ylabel(b, fontsize=11, fontweight="700", color=_mau(b))
        ax.set_yticks([]); ax.set_xlim(0, t[-1] + 1e-6)
        if i == 0:
            ax.set_title("Sóng âm")
        if i == len(bo) - 1:
            ax.set_xlabel("thời gian (giây)")

        ax = axes[i, 1]
        S = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=1024, hop_length=160)), ref=np.max)
        librosa.display.specshow(S, sr=16000, hop_length=160, x_axis="time", y_axis="hz",
                                 ax=ax, cmap="magma", vmin=-70, vmax=0)
        ax.set_ylim(0, 8000); ax.grid(False)
        ax.set_ylabel("Hz"); ax.set_xlabel("thời gian (giây)")
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.xaxis.set_major_locator(MaxNLocator(5))
        if i == 0:
            ax.set_title("Phổ thời gian – tần số")
    fig.suptitle(f"Cùng một câu · cùng một giọng — {cau} · {giong}",
                 fontsize=13.5, fontweight="700", y=1.015)
    _anh(fig, "Cùng câu · cùng giọng · ba bộ sinh",
         "Hình này là **hình đắt nhất để lên slide**. Nhìn cột phổ: vùng tần cao bị cắt cụt ở bộ sinh "
         "nào là dấu vết codec của bộ đó. Nhìn cột sóng âm: chỗ ngắt nghỉ khác nhau là dấu vết nhịp "
         "điệu. Cùng câu, cùng giọng mà hình khác nhau ⇒ đó chính là thứ mô hình sẽ học.")


# ═══════════════════════════════════════════════════ 6. ÂM HỌC
def ve_am_hoc(d, toi_da=8):
    ds = d[d.duong_dan.map(os.path.isfile)]
    if ds.empty:
        print("   (bỏ qua phần âm học — không tìm thấy tệp audio)")
        return
    import librosa
    bo = sorted(ds.bo_sinh.unique())
    ds = ds.groupby("bo_sinh", group_keys=False).head(toi_da * 60)
    print(f"   đọc {len(ds)} tệp để vẽ phổ…")

    SRD = 16000
    ltas, hf, roll, f0_all, mfcc_all = {}, {}, {}, {}, {}
    for b in bo:
        L, H, R, F, M = [], [], [], [], []
        for p in ds[ds.bo_sinh == b].duong_dan:
            try:
                y, sr = _doc(p)
            except Exception:
                continue
            if len(y) < sr * 0.4:
                continue
            y16 = librosa.resample(y, orig_sr=sr, target_sr=SRD) if sr != SRD else y
            S = np.abs(librosa.stft(y16, n_fft=1024, hop_length=160))
            if not S.size:
                continue
            L.append(S.mean(axis=1))
            freqs = librosa.fft_frequencies(sr=SRD, n_fft=1024)
            H.append(float(np.sum(S[freqs > SRD * 0.25] ** 2) / (np.sum(S ** 2) + 1e-12)))
            R.append(float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=SRD, roll_percent=0.95))))
            try:
                f0 = librosa.yin(y16, fmin=60, fmax=400, sr=SRD, frame_length=1024, hop_length=160)
                f0 = f0[np.isfinite(f0) & (f0 > 60) & (f0 < 400)]
                if f0.size > 8:
                    F.append(float(np.median(f0)))
            except Exception:
                pass
            m = librosa.feature.mfcc(y=y16, sr=SRD, n_mfcc=13, n_fft=1024, hop_length=160)
            M.append(np.r_[m.mean(axis=1), m.std(axis=1)])
        if L:
            ltas[b] = np.mean(np.stack(L), axis=0)
            hf[b], roll[b] = np.array(H), np.array(R)
            f0_all[b] = np.array(F)
            mfcc_all[b] = np.stack(M) if M else np.empty((0, 26))

    freqs = librosa.fft_frequencies(sr=SRD, n_fft=1024)

    fig, ax = plt.subplots(figsize=(10.5, 4))
    for b, m in ltas.items():
        db = 20 * np.log10(np.maximum(m, 1e-10)); ok = freqs > 60
        ax.plot(freqs[ok], db[ok], lw=2.1, color=_mau(b), label=b, alpha=0.92)
    ax.set_xscale("log"); ax.set_xlim(80, 8000)
    ax.set_xlabel("Tần số (Hz, thang log)"); ax.set_ylabel("Biên độ trung bình (dB)")
    ax.set_title("Phổ trung bình dài hạn (LTAS) — đo ở 16 kHz cho công bằng")
    ax.legend(frameon=False, loc="lower left")
    _anh(fig, "Phổ trung bình dài hạn (LTAS)",
         "Đường nào **tách ra** ở vùng 2–8 kHz là dấu vết vocoder. Đây là cơ sở vật lý của họ đặc "
         "trưng F2: năng lượng băng trên, độ dốc phổ, độ phẳng phổ.")

    if roll and hf:
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7))
        for ax, data, nhan, tieu in (
                (axes[0], roll, "Rolloff 95 % (Hz)", "Tần số dưới 95 % năng lượng"),
                (axes[1], hf, "% năng lượng trên 4 kHz", "Tỉ lệ năng lượng băng cao")):
            keys = list(data)
            vals = [data[k] * (100 if "%" in nhan else 1) for k in keys]
            bp = ax.boxplot(vals, patch_artist=True, widths=0.55,
                            medianprops=dict(color="#111", lw=2))
            for pc, k in zip(bp["boxes"], keys):
                pc.set_facecolor(_mau(k)); pc.set_alpha(0.68)
            ax.set_xticks(range(1, len(keys) + 1), keys, rotation=12)
            ax.set_ylabel(nhan); ax.set_title(tieu)
        _anh(fig, "Hai đặc trưng bắt dấu vết vocoder mạnh nhất",
             "Nếu hai hình này **tách rõ theo bộ sinh** thì mô hình sẽ đạt accuracy rất cao — nhưng "
             "đó có thể là học codec chứ không phải học 'sự nhân bản'. Phải kiểm bằng thí nghiệm "
             "chéo bộ sinh (E2).")

    if any(len(v) > 3 for v in f0_all.values()):
        fig, ax = plt.subplots(figsize=(9.5, 3.5))
        bins = 10
        for b, v in f0_all.items():
            if len(v) > 3:
                ax.hist(v, bins=bins, alpha=0.52, color=_mau(b), label=b, edgecolor="white")
        ax.set_xlabel("F0 trung vị (Hz)"); ax.set_ylabel("Số clip")
        ax.set_title("Cao độ cơ bản (F0) theo bộ sinh")
        ax.legend(frameon=False)
        _anh(fig, "Cao độ F0 — chất giọng có được bảo tồn?",
             "Với **cùng một giọng**, F0 phải giống nhau giữa các bộ sinh — vì cùng một người đọc. "
             "Lệch nhiều là **lỗi của bộ sinh**, không phải ưu điểm.")

    X, y = [], []
    for b, M in mfcc_all.items():
        if M.size:
            X.append(M); y += [b] * len(M)
    if X and sum(len(v) for v in X) > 8:
        from sklearn.decomposition import PCA
        from sklearn.preprocessing import StandardScaler
        X = np.vstack(X)
        Z = PCA(2).fit_transform(StandardScaler().fit_transform(X))
        ev = PCA(2).fit(StandardScaler().fit_transform(X)).explained_variance_ratio_
        fig, ax = plt.subplots(figsize=(7.8, 5))
        for b in bo:
            m = np.array(y) == b
            if m.any():
                ax.scatter(Z[m, 0], Z[m, 1], s=46, alpha=0.75, color=_mau(b), label=b,
                           edgecolor="white", linewidth=0.6)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.0f} % phương sai)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.0f} %)")
        ax.set_title("PCA trên MFCC mean+std (13 hệ số)")
        ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
        _anh(fig, "PCA — ba bộ sinh có tách thành cụm không?",
             "Cụm **tách rõ** ⇒ MFCC đủ phân biệt bộ sinh. Cụm **chồng lấn** ⇒ phải thêm họ đặc "
             "trưng khác (F2/F3), và đó là một phát hiện đáng viết vào báo cáo.")
    else:
        print("   (không đủ clip để vẽ PCA)")


# ═══════════════════════════════════ 7. NGHE & SO SÁNH (phần chính)
def muc_nghe(d, so_cau=4, kem_so=True, toi_da_mb=40.0):
    """Dựng HTML phần nghe: audio nhúng base64 + số đo từng clip."""
    ds = d[d.duong_dan.map(os.path.isfile)].copy()
    if ds.empty:
        return "<div class='warn'>Không tìm thấy tệp audio — bỏ qua phần nghe.</div>"

    bo = sorted(ds.bo_sinh.unique())
    giong = sorted(ds.giong.unique())

    # chọn câu có đủ mọi bộ sinh
    goc = ds.groupby(["cau_id"]).bo_sinh.nunique()
    du = sorted(goc[goc == len(bo)].index)
    if not du:
        return "<div class='warn'>Không có câu nào đủ mọi bộ sinh — bỏ qua phần nghe.</div>"
    chon = du[: max(1, so_cau)]

    # kiểm dung lượng trước
    tong_byte = 0
    for c in chon:
        for g in giong:
            for b in bo:
                p = ds[(ds.cau_id == c) & (ds.giong == g) & (ds.bo_sinh == b)]
                if len(p) and os.path.isfile(p.iloc[0].duong_dan):
                    tong_byte += os.path.getsize(p.iloc[0].duong_dan)
    mb = tong_byte * 1.37 / 1024 / 1024
    while mb > toi_da_mb and len(chon) > 1:
        chon = chon[:-1]
        tong_byte = 0
        for c in chon:
            for g in giong:
                for b in bo:
                    p = ds[(ds.cau_id == c) & (ds.giong == g) & (ds.bo_sinh == b)]
                    if len(p) and os.path.isfile(p.iloc[0].duong_dan):
                        tong_byte += os.path.getsize(p.iloc[0].duong_dan)
        mb = tong_byte * 1.37 / 1024 / 1024

    print(f"   nhúng audio: {len(chon)} câu × {len(giong)} giọng × {len(bo)} bộ sinh "
          f"≈ {mb:.1f} MB")

    out = []
    A = out.append

    # ---- so sánh nhanh: 1 câu × 1 giọng, đủ 3 bộ sinh ----
    c0, g0 = chon[0], giong[0]
    A("<div class='qc'>")
    A(f"<div class='qc-h'>So sánh nhanh — câu <code>{c0}</code> · giọng <code>{g0}</code></div>")
    A("<div class='nghe'>")
    for b in bo:
        sel = ds[(ds.cau_id == c0) & (ds.giong == g0) & (ds.bo_sinh == b)]
        if sel.empty:
            continue
        p = sel.iloc[0].duong_dan
        try:
            src = _nhung_bytes(p)
        except Exception:
            continue
        A("<div class='o' style='border-color:%s'>" % _mau(b))
        A(f"<div class='ten' style='color:{_mau(b)}'>{b}</div>")
        A(f"<audio controls preload='none' src='{src}'></audio>")
        if kem_so:
            try:
                y, sr = _doc(p)
                f0, roll, hf = _do_clip(y, sr)
                A("<div class='badges'>")
                A(f"<span class='b'>{len(y)/sr:.2f} s</span>")
                A(f"<span class='b'>F0 {f0:.0f} Hz</span>")
                A(f"<span class='b'>rolloff {roll:.0f} Hz</span>")
                A(f"<span class='b'>băng cao {hf:.1f} %</span>")
                A("</div>")
            except Exception:
                pass
        A("</div>")
    A("</div>")
    A("<p class='hint'>Bấm từng nút phát để nghe cùng một câu, cùng một giọng, do ba bộ sinh đọc. "
      "Câu hỏi cần trả lời bằng tai: <b>OmDiOm-ft có khác OmniVoice gốc không?</b> "
      "Nếu nghe giống hệt thì 4.000 bước fine-tune chưa tạo khác biệt đáng kể.</p>")
    A("</div>")

    # ---- bảng đầy đủ theo từng câu ----
    for c in chon:
        A(f"<div class='khoi'><div class='khoi-h'>Câu <code>{c}</code> — {len(giong)} giọng × "
          f"{len(bo)} bộ sinh</div>")
        A("<table class='bang nghe-bang'><thead><tr><th style='width:130px'>Giọng</th>")
        for b in bo:
            A(f"<th style='color:{_mau(b)}'>{b}</th>")
        A("</tr></thead><tbody>")
        for g in giong:
            A(f"<tr><td><code>{g}</code></td>")
            for b in bo:
                sel = ds[(ds.cau_id == c) & (ds.giong == g) & (ds.bo_sinh == b)]
                if sel.empty:
                    A("<td class='thieu'>—</td>")
                    continue
                p = sel.iloc[0].duong_dan
                try:
                    src = _nhung_bytes(p)
                    giay = float(sel.iloc[0].giay)
                    A(f"<td><audio controls preload='none' src='{src}'></audio>"
                      f"<span class='giay'>{giay:.2f} s</span></td>")
                except Exception:
                    A("<td class='thieu'>lỗi</td>")
            A("</tr>")
        A("</tbody></table></div>")

    A("<p class='hint'>Gợi ý khi nghe để chấm chất lượng nhân bản: <b>giọng có đúng người không</b> "
      "(so 7 dòng với nhau), <b>câu có rõ chữ không</b>, <b>cuối câu có bị nuốt không</b>, "
      "và <b>có tiếng lạ / rè không</b>.</p>")
    return "\n".join(out)


# ═══════════════════════════════════════════════════════════════ CSS
CSS = """
:root{--ink:#111820;--mut:#5a6673;--line:#e2e8ef;--bg:#f6f8fb;--acc:#2E86AB}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
font:16px/1.65 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,sans-serif}
.wrap{max-width:1180px;margin:0 auto;padding:0 22px 90px}
header{background:linear-gradient(135deg,#0f2c3a,#1b5a71 52%,#2E86AB);color:#fff;padding:52px 22px 44px}
header .in{max-width:1180px;margin:0 auto}
h1{margin:0 0 8px;font-size:33px;font-weight:750;letter-spacing:-.02em}
header p{margin:0;opacity:.92;font-size:15px}
.tags{margin-top:16px;display:flex;gap:9px;flex-wrap:wrap}
.tag{background:rgba(255,255,255,.16);border:1px solid rgba(255,255,255,.3);
padding:5px 12px;border-radius:100px;font-size:13px}
.kpi{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:13px;margin:-56px 0 36px}
.card{background:#fff;border:1px solid var(--line);border-radius:14px;padding:18px 20px;
box-shadow:0 3px 16px rgba(14,40,60,.08)}
.kpi .card{text-align:center;padding:19px 10px}
.kpi .n{font-size:26px;font-weight:780;color:var(--acc);letter-spacing:-.02em;line-height:1.1}
.kpi .l{font-size:12px;color:var(--mut);margin-top:5px;text-transform:uppercase;letter-spacing:.06em}
h2{font-size:13px;text-transform:uppercase;letter-spacing:.11em;color:var(--acc);
margin:48px 0 16px;padding-bottom:9px;border-bottom:2px solid var(--line);font-weight:750}
figure{margin:0 0 24px;background:#fff;border:1px solid var(--line);border-radius:14px;
padding:18px 20px 16px;box-shadow:0 2px 10px rgba(14,40,60,.05)}
figure img{width:100%;height:auto;display:block;border-radius:6px}
figure>div img{width:100%;height:auto;display:block}
figcaption{margin-top:13px;font-size:14.5px;color:var(--mut);border-left:3px solid var(--acc);padding-left:13px}
figcaption b{color:var(--ink);font-weight:680}
table.bang{width:100%;border-collapse:collapse;font-size:13.5px}
table.bang th{background:#eaf1f6;text-align:left;padding:9px 11px;font-weight:680;
border-bottom:2px solid var(--line)}
table.bang td{padding:8px 11px;border-bottom:1px solid var(--line);vertical-align:middle}
table.bang tr:nth-child(even) td{background:#fafcfe}
.warn,.demo{background:#fff8e6;border:1px solid #f0d78a;border-left:4px solid #e0a63a;
border-radius:10px;padding:14px 18px;margin:0 0 24px;font-size:14.5px}
.demo{background:#fdecec;border-color:#efb6b6;border-left-color:#c0392b;color:#7c1f1f;font-weight:600}
.qc,.khoi{background:#fff;border:1px solid var(--line);border-radius:14px;padding:18px 20px;
margin:0 0 22px;box-shadow:0 2px 10px rgba(14,40,60,.05)}
.qc{border-width:2px;border-color:#cfe0ea}
.qc-h,.khoi-h{font-size:15px;font-weight:720;margin-bottom:14px;color:#12303f}
.nghe{display:grid;grid-template-columns:repeat(auto-fit,minmax(255px,1fr));gap:14px}
.o{border:2px solid var(--line);border-radius:12px;padding:11px 13px;background:#fbfdfe}
.o .ten{font-size:12.5px;font-weight:780;text-transform:uppercase;letter-spacing:.07em;margin-bottom:7px}
audio{width:100%;height:34px;display:block}
.badges{display:flex;gap:6px;flex-wrap:wrap;margin-top:8px}
.b{font-size:11.5px;color:var(--mut);background:#eef3f7;border-radius:6px;padding:2px 7px}
.giay{font-size:11.5px;color:var(--mut);margin-left:6px}
.hint{font-size:14px;color:var(--mut);margin:2px 0 26px;line-height:1.6}
table.nghe-bang td{padding:7px 8px}
table.nghe-bang audio{height:30px;min-width:150px}
.thieu{color:#b0392b;font-weight:700;text-align:center}
code{background:#eef3f7;padding:1.5px 6px;border-radius:5px;font-size:13px}
footer{text-align:center;color:var(--mut);font-size:13px;margin-top:54px;
padding-top:18px;border-top:1px solid var(--line)}
@media(max-width:640px){.nghe{grid-template-columns:1fr}h1{font-size:26px}}
@media print{body{background:#fff}figure,.card,.qc,.khoi{box-shadow:none;break-inside:avoid}
header{-webkit-print-color-adjust:exact;print-color-adjust:exact}
.hint{display:none}}
"""


# ═══════════════════════════════════════════════════ dựng tệp
def tao_bao_cao(manifest, duong_ra="BAO_CAO.html", bang_cau=None,
                tieu_de="Báo cáo dữ liệu — EDA & nghe so sánh",
                so_cau_nghe=4, kem_so=True, so_anh_moi_bo=8,
                demo=False, im_lang=False):
    """Sinh tệp HTML tự chứa từ manifest.csv (+ tuỳ chọn bang_cau_chuan.csv)."""
    _HINH.clear()

    d = pd.read_csv(manifest) if isinstance(manifest, str) else manifest.copy()
    d.columns = [c.strip().lstrip("\ufeff") for c in d.columns]
    thieu = [c for c in CAN if c not in d.columns]
    if thieu:
        raise ValueError(f"manifest thiếu cột: {thieu}")

    if bang_cau is None:
        bang_cau = d.drop_duplicates("cau_id") if "thanh" in d.columns else None
    elif isinstance(bang_cau, str):
        b = pd.read_csv(bang_cau)
        b.columns = [c.strip().lstrip("\ufeff") for c in b.columns]
        bang_cau = b

    if not im_lang:
        print(f"  manifest: {len(d)} dòng")

    # ── vẽ EDA ──
    if not im_lang:
        print("  vẽ hình EDA…")
    ve_bo_cau(bang_cau)
    ve_day_du(d)
    ve_thoi_luong(d)
    ve_tan_so(d)
    ve_cung_cau(d)
    ve_am_hoc(d, toi_da=so_anh_moi_bo)

    # ── phần nghe ──
    if not im_lang:
        print("  nhúng audio…")
    html_nghe = muc_nghe(d, so_cau=so_cau_nghe, kem_so=kem_so)

    # ── ráp trang ──
    bo = sorted(d.bo_sinh.unique())
    P = []
    A = P.append
    A("<!doctype html><html lang='vi'><head><meta charset='utf-8'>")
    A("<meta name='viewport' content='width=device-width,initial-scale=1'>")
    A(f"<title>{tieu_de}</title><style>{CSS}</style></head><body>")

    A("<header><div class='in'>")
    A(f"<h1>{tieu_de}</h1>")
    A("<p>Nhóm 05 · Môn Nhận dạng mẫu · Đề tài 10 Voice Cloning · "
      "bài toán phân loại bộ sinh và giọng</p>")
    A("<div class='tags'>")
    for b in bo:
        A(f"<span class='tag'>{b}</span>")
    A(f"<span class='tag'>{d.giong.nunique()} giọng</span>")
    A(f"<span class='tag'>{d.cau_id.nunique()} câu</span>")
    A(f"<span class='tag'>{datetime.now():%d/%m/%Y}</span>")
    A("</div></div></header>")

    A("<div class='wrap'>")
    if demo:
        A("<div class='demo'>⚠️ BẢN DEMO — số liệu và audio là dữ liệu giả lập để kiểm tra code, "
          "không phải kết quả dự án.</div>")

    kpi = [(f"{len(d):,}", "clip"), (f"{d.cau_id.nunique()}", "câu"),
           (f"{d.giong.nunique()}", "giọng"), (f"{len(bo)}", "bộ sinh"),
           (f"{d.giay.sum()/60:.1f}", "phút audio"), (f"{d.giay.mean():.2f}s", "TB mỗi clip")]
    A("<div class='kpi'>")
    for n, l in kpi:
        A(f"<div class='card'><div class='n'>{n}</div><div class='l'>{l}</div></div>")
    A("</div>")

    warns = []
    srs = {b: int(d[d.bo_sinh == b].sr.mode().iloc[0]) for b in bo}
    if len(set(srs.values())) > 1:
        warns.append("Các bộ sinh dùng <b>tần số lấy mẫu khác nhau</b> "
                     f"({', '.join(f'{b}={v:,}' for b, v in srs.items())}) ⇒ bắt buộc resample trước "
                     "khi trích đặc trưng, nếu không kết quả bị nhiễu bởi băng thông (thí nghiệm E4).")
    if d.giay.min() < 0.5:
        warns.append(f"Có <b>{int((d.giay < 0.5).sum())}</b> clip dưới 0,5 giây — nên loại.")
    if len(bo) < 3:
        warns.append(f"Chỉ có <b>{len(bo)} bộ sinh</b> — bài toán 3 lớp chưa đủ.")
    _m = d.groupby("bo_sinh").size()
    if len(_m) and _m.min() != _m.max():
        warns.append("Số clip <b>không đều</b> giữa các bộ sinh — xem lưới độ đầy đủ.")
    if warns:
        A("<div class='warn'><b>Cần lưu ý</b><ul style='margin:10px 0 0 18px'>")
        for w in warns:
            A(f"<li>{w}</li>")
        A("</ul></div>")

    # ── NGHE trước ──
    A("<h2>Nghe &amp; so sánh</h2>")
    A(html_nghe)

    # ── EDA theo nhóm ──
    nhom = {"Bộ câu": [], "Độ đầy đủ": [], "Thời lượng & nhịp điệu": [],
            "Tần số lấy mẫu": [], "Âm học": []}
    for h in _HINH:
        t = h["tieu_de"].lower()
        if "cùng câu" in t:
            nhom["Âm học"].append(h)
        elif ("thanh" in t and "câu" in t) or "dài câu" in t or "train" in t:
            nhom["Bộ câu"].append(h)
        elif "lưới" in t or "bảng đếm" in t:
            nhom["Độ đầy đủ"].append(h)
        elif "tần số lấy mẫu" in t:
            nhom["Tần số lấy mẫu"].append(h)
        elif any(k in t for k in ("thời lượng", "tốc độ", "nhịp")):
            nhom["Thời lượng & nhịp điệu"].append(h)
        else:
            nhom["Âm học"].append(h)

    for ten, ds in nhom.items():
        if not ds:
            continue
        A(f"<h2>{ten}</h2>")
        for h in ds:
            A("<figure>")
            if "html" in h:
                A(h["html"])
            else:
                r = h.get("rong") or "100%"
                style = "" if r == "100%" else f" style='width:{r};margin:0 auto'"
                A(f"<div{style}><img src='data:image/png;base64,{h['b64']}' alt=''></div>")
            A(f"<figcaption><b>{h['tieu_de']}.</b> {h['doc_ra']}</figcaption>")
            A("</figure>")

    A("<footer>")
    A(f"Sinh tự động bởi <code>bao_cao_html.py</code> · {len(_HINH)} hình/bảng · "
      f"{datetime.now():%d/%m/%Y %H:%M}</footer>")
    A("</div></body></html>")

    html = "\n".join(P)
    with open(duong_ra, "w", encoding="utf-8") as f:
        f.write(html)
    if not im_lang:
        print(f"  📄 {duong_ra} · {len(html)/1024/1024:.2f} MB · {len(_HINH)} hình/bảng")
    return duong_ra


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("manifest")
    ap.add_argument("--ra", default="BAO_CAO.html")
    ap.add_argument("--bang-cau", default=None)
    ap.add_argument("--so-cau-nghe", type=int, default=4)
    ap.add_argument("--demo", action="store_true")
    a = ap.parse_args()
    tao_bao_cao(a.manifest, a.ra, a.bang_cau, so_cau_nghe=a.so_cau_nghe, demo=a.demo)

## Xuất bản Báo cáo Đa phương tiện và Lưu trữ Tài nguyên Thực nghiệm
Khởi tạo tệp báo cáo nghiên cứu hoàn chỉnh `BAO_CAO.html` tự chứa dữ liệu và đóng gói tập tin nén lưu trữ phục vụ các bước trích xuất đặc trưng tiếp theo.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Ô 11 · TẠO BÁO CÁO HTML + ĐÓNG GÓI
# ═══════════════════════════════════════════════════════════════════════════
import importlib, sys, os

# nạp lại module mỗi lần chạy để sửa code có hiệu lực ngay
if "bao_cao_html" in sys.modules:
    del sys.modules["bao_cao_html"]
for _p in (os.getcwd(), "/kaggle/working", os.path.dirname(RA)):   # notebook chạy ở đâu cũng tìm thấy module
    if _p not in sys.path:
        sys.path.insert(0, _p)
import bao_cao_html
importlib.reload(bao_cao_html)

MANIFEST = os.path.join(RA, "manifest.csv")
HTML     = os.path.join(os.path.dirname(RA), "BAO_CAO.html")

# bảng câu (để vẽ 3 hình phần "Bộ câu")
import pandas as pd
BANG_CAU = pd.DataFrame(CAU, columns=["cau_id", "noi_dung", "thanh", "tap", "so_am_tiet"])

# số câu đưa vào phần nghe — 3 câu × 7 giọng × 3 bộ sinh = 63 trình phát (~6 MB)
SO_CAU_NGHE = 3 if not CHAY_THU else 1

print("Đang dựng báo cáo…")
bao_cao_html.tao_bao_cao(
    manifest  = MANIFEST,
    duong_ra  = HTML,
    bang_cau  = BANG_CAU,
    so_cau_nghe = SO_CAU_NGHE,
    kem_so    = True,
    so_anh_moi_bo = 8,
)
print(f"\n✅ {HTML} · {os.path.getsize(HTML)/1024/1024:.1f} MB")
print("   Tải về → mở bằng Chrome → Ctrl+P → Save as PDF để chèn slide")

# ── đóng gói ──────────────────────────────────────────────────────────────
ten = os.path.join("/kaggle/working", "pa2_buoc1_" + time.strftime("%m%d_%H%M"))
shutil.make_archive(ten, "zip", RA)
print(f"\n📦 {ten}.zip · {os.path.getsize(ten + '.zip')/1024/1024:.0f} MB")

print("""
════════════════════════════════════════════════════════════════
 BƯỚC TIẾP
   1. Save Version (Commit) để giữ output
   2. Tải 3 tệp: BAO_CAO.html · pa2/manifest.csv · pa2_buoc1_*.zip
   3. Gửi tôi manifest.csv — tôi dựng Bước 2:
      trích F1/F2/F3 → 6 mô hình → ma trận nhầm lẫn 3×3 và 7×7
════════════════════════════════════════════════════════════════
""")
for b in BO_SINH:
    n = len(glob.glob(os.path.join(DIR_AM, b, "*", f"*.{DINH_DANG}")))
    print(f"  audio/{b:10s} {n:5d} clip")